In [1]:
from IPython.display import HTML as NotebookHTML, display
from datetime import datetime
from tts_data_utils.core.data_item import DataItem
from tts_data_utils.core.data_container import DataContainer



# TTS Data Utils Demo
This is a simple demo of TTS Data Utils. It is one of the most important repositories in our codebase as many other repositories require it. 

In a nutshell, TTS Data Utils is for managing 2D data. We provide two core abstract base classes, DataContainer and DataItem. These are meant to be extendeed by developers for specific use cases on a given project. We provide some multimission data containers like EHA and EVRs, but there will inevitably be other items needed by each product such as bespoke planning inputs from other teams (something like a science team input CSV).

Here we demonstrate a very simple extension for fictional spacecraft data.

In [2]:
class TelemetryItem(DataItem):
    # Define valid keys and their types for validation
    DICT_VALID_KEYS = [
        ('timestamp', datetime),
        ('sensor_id', str),
        ('value', float),
        ('status', str)
    ]
    
    # Define how times are parsed/printed
    TIME_FORMATS = {'timestamp': '%Y-%m-%dT%H:%M:%S'}

    @property
    def time(self):
        return self['timestamp']

    @property
    def default_html_cell_styles(self):
        """Custom Highlighting: Red background if value > 90."""
        styles = {k: {} for k in self.values.keys()}
        if self['value'] > 90.0:
            styles['value'] = {'background-color': '#ffcccc', 'color': 'red', 'font-weight': 'bold'}
        return styles

    @property
    def default_html_row_style(self):
        if self['sensor_id'] == 'BATT_0A':
            return {'color': 'blue'}
        else:
            return {}

class TelemetryContainer(DataContainer):
    DATA_ITEM_CLS = TelemetryItem
    NAME = 'spacecraft_telemetry'

    @property
    def default_time_label(self):
        return 'timestamp'

# Adding Data

The simplest way to provide data to a DataContainer is via a list of record dicts passed to the raw_data kwarg as demonstrated here. There is also a csv_path kwarg that can be used to point to a file.

In [3]:
data = [
    {'timestamp': '2023-10-01T12:00:00', 'sensor_id': 'TEMP_01', 'value': 72.5, 'status': 'NOMINAL'},
    {'timestamp': '2023-10-01T12:05:00', 'sensor_id': 'TEMP_01', 'value': 95.2, 'status': 'CRITICAL'},
    {'timestamp': '2023-10-01T12:10:00', 'sensor_id': 'TEMP_01', 'value': 88.0, 'status': 'WARNING'},
    {'timestamp': '2023-10-01T12:00:00', 'sensor_id': 'BATT_0A', 'value': 98.1, 'status': 'NOMINAL'},
    {'timestamp': '2023-10-01T12:05:00', 'sensor_id': 'BATT_0A', 'value': 45.0, 'status': 'LOW'},
]

# Initialize container (set cast_fields=True to handle string-to-datetime conversion)
tm = TelemetryContainer(raw_data=data, cast_fields=True)

# Power Table
One of the nicest features out of the box with DataContainers is the power_table method. This leverages the PowerTable class in tts_html_utils, and quickly and easily turns your 2D data into something visualizable in Jupyter, an HTML report, or a web page.

Note the html style properties defined above. These control the style of the tables produced by power_table()

In [4]:
# This will render with our custom red highlights for values > 90
pt = tm.power_table(
    id='telemetry_table',
    superheader="Spacecraft Telemetry Report"
)
NotebookHTML(pt.render())

# Filtering and Sorting

Another powerful utility provided by DatContainer is filtering and sorting. While many users have done this historically with list comprehensions and that's easy enough for one or two lines of code, we find that the mental fatigue on developers juggling many comprehensions in a complex codebase is just too high, so we've abstracted them into various filter methods. Here we have demonstrated `gt` and `eq`, but there are all of the typical ones including looking for values in a list `isin` or `notin`, doing regular expressions on the cell values `matches` and `doesnotmatch`, filtering based on time `before`, `after`, `between` or reporting only values differnt than the last one `onchange`. See the docs for DataContainer for more information.



In [5]:
# 1. Filter for high values
high_values = tm.gt('value', 80.0)

# 2. Filter for specific sensor and sort by value
batt_sorted = tm.eq('sensor_id', 'BATT_0A').sort(by='value')

print(f"Total records: {len(tm)}")
print(f"High value records: {len(high_values)}")
NotebookHTML(high_values.power_table().render())

Total records: 5
High value records: 3


Note that you don't actually need to call power_table in jupyter since it's part of the _repr_html_ method.

In [6]:
batt_sorted

+---------------------+-------------+---------+----------+
| timestamp           | sensor_id   |   value | status   |
+=====================+=============+=========+==========+
| 2023-10-01T12:05:00 | BATT_0A     |    45   | LOW      |
+---------------------+-------------+---------+----------+
| 2023-10-01T12:00:00 | BATT_0A     |    98.1 | NOMINAL  |
+---------------------+-------------+---------+----------+

# Visual Diff

The final out of the box feature we will demonstrate here is the visual diff. Given two containers of the same type, the visual_diff method along with the VisualDiff `tts_html_utils` component provide a quick and easy way to visually compare the two.

Also note that there is a simpler `diff` method on each container that works similarly, but is less well optimzed for visualization. It is very nice to have for unit testing.

In [7]:
from tts_html_utils.core.compiler import HtmlCompiler
from tts_html_utils.core.components import H1, P, HR
from tts_html_utils.visdiff.visdiff import VisualDiff

data_v2 = [
    {'timestamp': '2023-10-01T12:10:00', 'sensor_id': 'BATT_0B', 'value': 50.0, 'status': 'NEW'},     # New row
    {'timestamp': '2023-10-01T12:00:00', 'sensor_id': 'TEMP_01', 'value': 72.5, 'status': 'NOMINAL'},
    {'timestamp': '2023-10-01T12:05:00', 'sensor_id': 'TEMP_01', 'value': 99.9, 'status': 'CRITICAL'}, # Changed value
    # Row 3 deleted
    {'timestamp': '2023-10-01T12:00:00', 'sensor_id': 'BATT_0A', 'value': 98.1, 'status': 'NOMINAL'},
    {'timestamp': '2023-10-01T12:05:00', 'sensor_id': 'BATT_0A', 'value': 45.0, 'status': 'LOW'}
]
tm_v2 = TelemetryContainer(raw_data=data_v2, cast_fields=True)
# 1. Generate the aligned diff data using data_utils logic
# This produces two VisualDiffContainers (left and right)

left_diff_container, right_diff_container = tm.visual_diff(tm_v2)

# 2. Convert containers into PowerTable components
# We MUST call .power_table() here because VisualDiff needs the table IDs 
# to inject alignment JavaScript
left_table = left_diff_container.power_table(id='left-tm-table')
right_table = right_diff_container.power_table(id='right-tm-table')

# 3. Create the VisualDiff side-by-side component
# This will align the rows from the two tables
vd_component = VisualDiff(
    left_table, 
    right_table, 
    left_label="Baseline Telemetry", 
    right_label="Comparison Telemetry"
)

NotebookHTML(vd_component.render())

# Lorem
For developers we have also provided the Lorem keyword. This is to enable develpers to quickly test using representative data with the corret types, even if the data itself is nonsense.

Note also here that when working in jupyter, we don't actually _need_ to call the power_table method to visualize the tables because they have the right repr methods defined on them. They will also print nicely at the command line if you drop into a trace.

In [8]:
from tts_data_utils.multimission.evr import EvrContainer
from tts_data_utils.multimission.eha import EhaContainer
from tts_data_utils.core.generic import GenericContainer

eha_container = EhaContainer(lorem=3)
generic_container = GenericContainer(lorem=4)

In [9]:
eha_container

+-------------------------------------------------------+-------------+----------------------------------------+-------------+---------+--------+-------------------+--------------------------------------------------+--------------------------+--------------------------+---------------------+----------------------------+-------------+--------------+------------------------------------------+-------+-------------+-----------------------------------------------------+---------------------------------+------------+-----------------------------------------------------------+
| recordType                                            |   sessionId | sessionHost                            | channelId   |   dssId |   vcid | name              | module                                           | ert                      | scet                     | rct                 | lst                        |        sclk | dn           | dnStr                                    |    eu | status      | dnAlarmState                                        | euAlarmState                    | realtime   | type                                                      |
+=======================================================+=============+========================================+=============+=========+========+===================+==================================================+==========================+==========================+=====================+============================+=============+==============+==========================================+=======+=============+=====================================================+=================================+============+===========================================================+
| Aliquip lorem nostrud consequat elit irure ex ullamco |        7390 | Pariatur ea nostrud sit officia nulla  | ID-6358     |    4761 |   9266 | Qui               | Ad dolore consequat id minim irure enim aute     | 2025-337T06:01:24.000000 | 2025-039T06:01:24.000000 | 2025-05-15 06:01:24 | 2025-03-05T06:01:24.625699 | 8.60725e+06 | 195          | Nulla sunt aliqua quis ipsum magna       | 19.09 | error       | Enim cupidatat minim qui quis labore occaecat irure | Sunt amet commodo aliqua        | True       | Enim ut commodo et dolor                                  |
+-------------------------------------------------------+-------------+----------------------------------------+-------------+---------+--------+-------------------+--------------------------------------------------+--------------------------+--------------------------+---------------------+----------------------------+-------------+--------------+------------------------------------------+-------+-------------+-----------------------------------------------------+---------------------------------+------------+-----------------------------------------------------------+
| Nulla pariatur ipsum                                  |        4257 | Aliqua dolor elit ex est reprehenderit | ID-8697     |    3462 |   8277 | Magna minim anim  | Magna amet                                       | 2025-027T06:01:24.000000 | 2025-241T06:01:24.000000 | 2025-10-24 06:01:24 | 2025-03-03T06:01:24.625742 | 1.73883e+06 | Labore nulla | Ullamco fugiat quis dolore irure eiusmod |       | pending     | Amet quis lorem excepteur dolor ipsum commodo       | Ea incididunt commodo excepteur | True       | Enim cupidatat excepteur sint                             |
+-------------------------------------------------------+-------------+----------------------------------------+-------------+---------+--------+-------------------+--------------------------------------------------+--------------------------+--------------------------+---------------------+----------------------------+-------------+--------------+------------------------------------------+-------+-------------+-----------------------------------------------------+---------------------------------+------------+----------------------

In [10]:
generic_container

+------+----------------------------------+---------------------------------------------------------------------------------------------------------------------------+----------------------------+---------+---------+-------------+
|   id | title                            | description                                                                                                               | timestamp                  |   value |   count | status      |
+======+==================================+===========================================================================================================================+============================+=========+=========+=============+
|    1 | Labore velit                     | Aliqua nulla ut eiusmod commodo laborum amet proident elit esse ea enim magna sunt aliquip ullamco qui culpa exercitation | 2025-09-10 06:01:24.625917 |   37.03 |     808 | success     |
+------+----------------------------------+---------------------------------------------------------------------------------------------------------------------------+----------------------------+---------+---------+-------------+
|    2 | Cupidatat culpa mollit           | Ad labore ex dolor enim cillum sed consequat deserunt esse id do ut voluptate sint laboris veniam pariatur dolore aliquip | 2025-06-02 06:01:24.625925 |   10.8  |     795 | success     |
+------+----------------------------------+---------------------------------------------------------------------------------------------------------------------------+----------------------------+---------+---------+-------------+
|    3 | Sint cupidatat laboris culpa non | Magna deserunt do fugiat et commodo consectetur labore qui est quis pariatur id                                           | 2025-10-29 06:01:24.625932 |   24.52 |     386 | unknown     |
+------+----------------------------------+---------------------------------------------------------------------------------------------------------------------------+----------------------------+---------+---------+-------------+
|    4 | Nisi sunt duis                   | Et pariatur sed incididunt ut tempor aliquip qui cillum consequat reprehenderit sint do laboris aliqua                    | 2025-10-18 06:01:24.625937 |   92.97 |     839 | in_progress |
+------+----------------------------------+---------------------------------------------------------------------------------------------------------------------------+----------------------------+---------+---------+-------------+